In [0]:
import subprocess
import os
import sys
import tempfile
import traceback
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

def initialize_secrets():
    try:
        SECRET_SCOPE_NAME = os.environ.get("SECRET_SCOPE_NAME", "redox_oauth_keys")
        if not SECRET_SCOPE_NAME:
            raise ValueError("SECRET_SCOPE_NAME environment variable is required but was not set")
        print(f"[redox-proxy] Using secret scope: {SECRET_SCOPE_NAME}")
        print(f"[redox-proxy] Retrieving private key...")
        PRIVATE_KEY = w.secrets.get_secret(scope=SECRET_SCOPE_NAME, key="private_key").value
        print(f"[redox-proxy] Retrieving KID...")
        KID = w.secrets.get_secret(scope=SECRET_SCOPE_NAME, key="kid").value
        print(f"[redox-proxy] Retrieving client ID...")
        CLIENT_ID = w.secrets.get_secret(scope=SECRET_SCOPE_NAME, key="client_id").value
        os.environ["OAUTH_CLIENT_ID"] = CLIENT_ID
        os.environ["OAUTH_KEY_ID"] = KID
        print(f"[redox-proxy] Secrets retrieved and environment variables set")
        print(f"[redox-proxy] Writing private key to temp file...")
        temp_key_file = tempfile.NamedTemporaryFile(
            mode='w'
            , delete=False
            , suffix='.pem'
        )
        temp_key_file.write(PRIVATE_KEY)
        temp_key_file.close()
        os.chmod(temp_key_file.name, 0o600)
        os.environ["OAUTH_KEY_PATH"] = temp_key_file.name
        print(f"[redox-proxy] Private key written to: {temp_key_file.name}")
    except Exception as e:
        print(f"[redox-proxy] ERROR initializing secrets: {e}")
        print(f"[redox-proxy] Traceback: {traceback.format_exc()}")
        raise

initialize_secrets()

# Start the binary
proc = subprocess.Popen(
    ["/Volumes/mkgs/redox/bin/redox-mcp"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    env=os.environ.copy()
)

# Send initialize request
import json
request = {"jsonrpc": "2.0", "method": "initialize", "id": 1, "params": {"protocolVersion": "2025-11-25", "capabilities": {}}}
proc.stdin.write(json.dumps(request) + "\n")
proc.stdin.flush()

# Wait and check what comes back
import time
time.sleep(2)
print("Return code:", proc.poll())
print("Has stdout?", proc.stdout.readable())

# Try to read
import select
if select.select([proc.stdout], [], [], 1)[0]:
    print("STDOUT:", proc.stdout.readline())
else:
    print("No stdout available")